In [ ]:
from typing import Any, Callable, Sequence, Tuple, Dict
import time
import jax
import jax.numpy as jnp
from jax import random
import optax
import os
from flax import linen as nn
from flax.training import train_state, checkpoints
from tqdm import tqdm
import numpy as np
from PIL import Image
import tensorflow as tf

# -----------------------------
# Configuration / Hyperparams
# -----------------------------
NUM_CLASSES = 4
TASK_NAMES = ["denoise", "deblur", "derain", "dehaze", "enhance"]
PATCH_SIZE = 225
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 20
SEED = 42
LOG_EVERY = 10
CKPT_PATH = "./router_checkpoints"
DATA_PATH = "./dataset"

In [ ]:
# -----------------------------
# Utilities
# -----------------------------

def preprocess_pil(img: Image.Image, image_size: int = PATCH_SIZE) -> np.ndarray:
    """Convert PIL image to float32 numpy array shaped (H,W,3) normalized to [0,1]."""
    img = img.convert("RGB")
    img = img.resize((image_size, image_size), resample=Image.BILINEAR)
    arr = np.array(img).astype(np.float32) / 255.0
    # HWC format is standard for Flax/JAX
    return arr


def load_image(filepath):
    """Load and preprocess image."""
    img = Image.open(filepath).convert("RGB")
    img = np.asarray(img, np.float32) / 255.0
    return img


def make_shape_even(image):
    """Pad the image to have even shapes."""
    height, width = image.shape[0], image.shape[1]
    padh = 1 if height % 2 != 0 else 0
    padw = 1 if width % 2 != 0 else 0
    image = jnp.pad(image, [(0, padh), (0, padw), (0, 0)], mode="reflect")
    return image


def mod_padding_symmetric(image, factor=64):
    """Padding the image to be divided by factor."""
    height, width = image.shape[0], image.shape[1]
    height_pad, width_pad = (
        ((height + factor) // factor) * factor,
        ((width + factor) // factor) * factor,
    )
    padh = height_pad - height if height % factor != 0 else 0
    padw = width_pad - width if width % factor != 0 else 0
    image = jnp.pad(
        image, [(padh // 2, padh // 2), (padw // 2, padw // 2), (0, 0)], mode="reflect"
    )
    return image


def random_crop(image, crop_size):
    """Random crop for data augmentation."""
    h, w = image.shape[0], image.shape[1]

    if h > crop_size and w > crop_size:
        top = np.random.randint(0, h - crop_size)
        left = np.random.randint(0, w - crop_size)

        image = image[top : top + crop_size, left : left + crop_size, :]

    return image


def random_flip(image):
    """Random horizontal and vertical flip."""
    if np.random.rand() > 0.5:
        image = np.fliplr(image)

    if np.random.rand() > 0.5:
        image = np.flipud(image)

    return image


def random_rotation(image):
    """Random 90-degree rotation."""
    k = np.random.randint(0, 4)
    image = np.rot90(image, k=k)
    return image

In [ ]:
# -----------------------------
# Model: small DW-Conv backbone
# -----------------------------


class DepthwiseConv(nn.Module):
    kernel_size: Tuple[int, int]
    strides: Tuple[int, int] = (1, 1)
    padding: str = "SAME"

    @nn.compact
    def __call__(self, x):
        in_ch = x.shape[-1]
        # Flax conv expects (N, H, W, C) by default
        # We'll use `feature_group_count=in_ch` for depthwise
        x = nn.Conv(
            features=in_ch,
            kernel_size=self.kernel_size,
            strides=self.strides,
            padding=self.padding,
            feature_group_count=in_ch,
            use_bias=False,
        )(x)
        return x


class DwSepBlock(nn.Module):
    out_ch: int
    stride: int = 1

    @nn.compact
    def __call__(self, x, train: bool = True):
        # Depthwise
        x = DepthwiseConv(kernel_size=(3, 3), strides=(self.stride, self.stride))(x)
        x = nn.BatchNorm(use_running_average=not train)(x)
        x = nn.relu(x)
        # Pointwise
        x = nn.Conv(features=self.out_ch, kernel_size=(1, 1), use_bias=False)(x)
        x = nn.BatchNorm(use_running_average=not train)(x)
        x = nn.relu(x)
        return x


class SmallBackbone(nn.Module):
    """Small lightweight backbone producing a global-pooled embedding.

    Input shape: (N, H, W, C) with C=3
    Output: (N, embedding_dim)
    """

    embedding_dim: int = 576

    @nn.compact
    def __call__(self, x, train: bool = True):
        # x: NHWC
        assert x.ndim == 4
        # initial conv
        x = nn.Conv(
            features=32,
            kernel_size=(3, 3),
            strides=(2, 2),
            padding="SAME",
            use_bias=False,
        )(x)
        x = nn.BatchNorm(use_running_average=not train)(x)
        x = nn.relu(x)

        # a few depthwise separable blocks
        x = DwSepBlock(out_ch=64, stride=1)(x, train=train)
        x = DwSepBlock(out_ch=96, stride=2)(x, train=train)
        x = DwSepBlock(out_ch=160, stride=2)(x, train=train)
        x = DwSepBlock(out_ch=self.embedding_dim, stride=2)(x, train=train)

        # global avg pool
        # x shape: (N, H, W, C)
        x = x.mean(axis=(1, 2))  # (N, C)
        return x


class RouterHead(nn.Module):
    num_classes: int = NUM_CLASSES

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(256)(x)
        x = nn.relu(x)
        x = nn.Dense(128)(x)
        x = nn.relu(x)
        x = nn.Dense(self.num_classes)(x)
        return x


class RouterModel(nn.Module):
    embedding_dim: int = 576
    num_classes: int = NUM_CLASSES

    def setup(self):
        self.backbone = SmallBackbone(embedding_dim=self.embedding_dim)
        self.head = RouterHead(num_classes=self.num_classes)

    def __call__(self, x, train: bool = True):
        feats = self.backbone(x, train=train)
        logits = self.head(feats)
        return logits

In [ ]:
# -----------------------------
# Train state / Init / Trainer
# -----------------------------


class TrainState(train_state.TrainState):
    batch_stats: Any = None  # for BatchNorm

def create_train_state(rng, learning_rate=LEARNING_RATE):
    model = RouterModel()
    input_shape = (1, PATCH_SIZE, PATCH_SIZE, 3)
    variables = model.init(rng, jnp.ones(input_shape, jnp.float32), train=True)
    params = variables["params"]
    batch_stats = variables.get("batch_stats")

    tx = optax.adamw(learning_rate=learning_rate, weight_decay=WEIGHT_DECAY)
    state = TrainState.create(
        apply_fn=model.apply, params=params, tx=tx, batch_stats=batch_stats
    )
    return state


# -----------------------------
# Loss / metrics
# -----------------------------


def cross_entropy_loss(logits, labels):
    onehot = jax.nn.one_hot(labels, logits.shape[-1])
    loss = optax.softmax_cross_entropy(logits=logits, labels=onehot)
    return loss.mean()


@jax.jit
def compute_metrics(logits, labels):
    loss = cross_entropy_loss(logits, labels)
    acc = jnp.mean(jnp.argmax(logits, -1) == labels)
    return {"loss": loss, "accuracy": acc}


# -----------------------------
# Training / Eval step
# -----------------------------

@jax.jit
def train_step(state: TrainState, batch: Tuple[jnp.ndarray, jnp.ndarray]):
    imgs, labels = batch  # imgs: NHWC float32, labels: (N,)

    def loss_fn(params):
        logits, new_model_state = state.apply_fn(
            {"params": params, "batch_stats": state.batch_stats},
            imgs,
            train=True,
            mutable=["batch_stats"],
        )
        loss = cross_entropy_loss(logits, labels)
        return loss, (logits, new_model_state)

    grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
    (loss, (logits, new_model_state)), grads = grad_fn(state.params)
    state = state.apply_gradients(
        grads=grads, batch_stats=new_model_state["batch_stats"]
    )
    metrics = compute_metrics(logits, labels)
    return state, metrics

def train_epoch(state: TrainState, train_dataset, epoch):
    """Train for one epoch."""
    batch_metrics = []
    print(f"Starting training epoch {epoch + 1}")
    rng, init_rng = jax.random.split(jax.random.PRNGKey(SEED))
    for step, (images, labels) in enumerate(train_dataset):
        rng, step_rng = jax.random.split(rng)
        images = jnp.array(images)
        labels = jnp.array(labels)
        if images.shape != labels.shape:
            print(f"Skipping step {step} due to shape mismatch: input {images.shape}, target {labels.shape}")
            continue

        rng, step_rng = jax.random.split(rng)
        state, metrics = train_step(state, images, labels, step_rng)
        batch_metrics.append(metrics)

        if (step + 1) % LOG_EVERY == 0:
            metrics_np = jax.device_get(metrics)
            print(
                f"Epoch {epoch + 1}, Step {step + 1}: "
                f'loss = {metrics_np["loss"]:.4f}, '
                f'accuracy = {metrics_np["accuracy"]:.2f}'
            )

    # Compute epoch metrics
    epoch_metrics = {
        k: np.mean([m[k] for m in batch_metrics]) for k in batch_metrics[0].keys()
    }

    return state, epoch_metrics


@jax.jit
def eval_step(state: TrainState, images: jnp.ndarray, labels: jnp.ndarray):
    variables = {"params": state.params, "batch_stats": state.batch_stats}
    logits = state.apply_fn(variables, images, train=False)
    metrics = compute_metrics(logits, labels)
    return metrics

def evaluate(state, val_dataset):
    """Evaluate on validation set."""
    batch_metrics = []
    
    # Use tqdm for progress bar
    pbar = tqdm(val_dataset, desc="Evaluating")
    expected_batch_size = BATCH_SIZE
    for images, labels in pbar:
        images = jnp.asarray(images, dtype=jnp.float32)
        labels = jnp.asarray(labels, dtype=jnp.int32)
        
        current_batch_size = images.shape[0]            
        # Pad if necessary to avoid recompilation
        if current_batch_size < expected_batch_size:
            pad_amount = expected_batch_size - current_batch_size
            pad_width = [(0, pad_amount)] + [(0, 0)] * (images.ndim - 1)
            # Use edge padding for target to avoid huge errors
            images = jnp.pad(images, pad_width, mode='edge')
            labels = jnp.pad(labels, pad_width, mode='edge')
            
        metrics = eval_step(state, images, labels)
        metrics_np = jax.device_get(metrics)
        batch_metrics.append((metrics_np, current_batch_size))

    # Compute average metrics
    if not batch_metrics:
        return {}
        
    avg_metrics = {}
    total_samples = sum(count for _, count in batch_metrics)
    
    # Get keys from first batch
    keys = batch_metrics[0][0].keys()
    
    for k in keys:
        # Weighted average
        weighted_sum = sum(m[k] * count for m, count in batch_metrics)
        avg_metrics[k] = weighted_sum / total_samples

    return avg_metrics


# -----------------------------
# Dataset helpers
# -----------------------------

def read_lines_from_file(basepath, task, is_training):
    file = "train.txt" if is_training else "test.txt"
    filepath = os.path.join(basepath, task , file)
    with open(filepath, "r", encoding="utf-8") as f:
        paths = [line.strip() for line in f if line.strip()]

    imgs_dir = os.path.join(basepath, task, "imgs")
    for path in paths:
        p = os.path.join(imgs_dir, path)
        if os.path.exists(p):
            yield p


def create_dataset(data_dir, batch_size, is_training=True):
    """Create TensorFlow dataset for training/validation."""
    print(
        f"Creating {'training' if is_training else 'validation'} dataset from {data_dir}"
    )
    images = []
    labels = []

    for task in TASK_NAMES:
        img_paths = read_lines_from_file(data_dir, task, is_training)
        for img_path in img_paths:
            images.append(img_path)
            labels.append(task)

    def load_and_preprocess(input_path, label):
        """Load and preprocess a single pair of images."""
        input_img = preprocess_pil(Image.open(input_path.numpy().decode()))
        # Padding images to have even shapes
        input_img = make_shape_even(input_img)
        # Padding images to be multiples of 64
        input_img = mod_padding_symmetric(input_img, factor=64)

        if is_training:
            # Data augmentation
            input_img = random_crop(input_img, PATCH_SIZE)
            input_img = random_flip(input_img)
            input_img = random_rotation(input_img)

        return (
            input_img.astype(np.float32),
            tf.cast(label, tf.int32),
        )

    dataset = tf.data.Dataset.from_tensor_slices((images, labels))

    if is_training:
        dataset = dataset.shuffle(buffer_size=1000)

    dataset = dataset.map(
        lambda x, y: tf.py_function(
            func=load_and_preprocess,
            inp=[x, y],
            Tout=[tf.float32, tf.int32],
        ),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    dataset = dataset.batch(batch_size, drop_remainder=is_training)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset, len(images)


def data_loader(
    images: np.ndarray, labels: np.ndarray, batch_size=BATCH_SIZE, shuffle=True
):
    n = len(images)
    idxs = np.arange(n)
    if shuffle:
        np.random.shuffle(idxs)
    for i in range(0, n, batch_size):
        batch_idx = idxs[i : i + batch_size]
        yield images[batch_idx], labels[batch_idx]

# -----------------------------
# Saving / Loading
# -----------------------------


def save_checkpoint(state: TrainState, step: int):
    save_dict = {
        "params": state.params,
        "batch_stats": state.batch_stats,
        "opt_state": state.opt_state,
    }
    checkpoints.save_checkpoint(
        ckpt_dir=CKPT_PATH, target=save_dict, step=step, overwrite=True
    )


def load_checkpoint(state: TrainState):
    ckpt = checkpoints.restore_checkpoint(ckpt_dir=CKPT_PATH, target=None)
    if ckpt:
        state = state.replace(
            params=ckpt["params"], batch_stats=ckpt.get("batch_stats")
        )
    return state


# -----------------------------
# Inference helper
# -----------------------------


def route_image_pil(
    state: TrainState, pil_img: Image.Image, rng=None
) -> Tuple[str, float]:
    arr = preprocess_pil(pil_img)  # HWC float32
    arr = jnp.array(arr)
    arr = arr[None, ...]  # 1HWC
    variables = {"params": state.params, "batch_stats": state.batch_stats}
    logits = state.apply_fn(variables, arr, train=False, mutable=False)
    probs = jax.nn.softmax(logits, axis=-1)
    probs = np.array(probs[0])
    idx = int(np.argmax(probs))
    return TASK_NAMES[idx], float(probs[idx])


In [ ]:
# -----------------------------
# Training Loop
# -----------------------------

rng = random.PRNGKey(SEED)
state = create_train_state(rng)

train_dataset, train_size = create_dataset(
    data_dir=DATA_PATH, batch_size=BATCH_SIZE, is_training=True
)
test_dataset, test_size = create_dataset(
    data_dir=DATA_PATH, batch_size=BATCH_SIZE, is_training=False
)

best_acc = 0.0
print(f"Training for {range(NUM_EPOCHS)} epochs...")
for epoch in range(NUM_EPOCHS):
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    state, train_metrics = train_epoch(state, train_dataset, epoch)

    print(
        f"Epoch {epoch} training: "
        f'loss = {train_metrics["loss"]:.4f}, '
        f'accuracy = {train_metrics["accuracy"]:.2f}'
    )

    # Validation
    val_metrics = evaluate(state, test_dataset.take(15))
    print(
        f"Epoch {epoch} validation: "
        f'loss = {val_metrics["loss"]:.4f}, '
        f'accuracy = {val_metrics["accuracy"]:.2f}'
    )

    if (epoch + 1) % 10 == 0 or val_metrics["accuracy"] > best_acc:
        ckpt_path = os.path.join(CKPT_PATH, f"checkpoint_{epoch}")
        checkpoints.save_checkpoint(
            ckpt_dir=ckpt_path,
            target={"opt": {"target": state.params}},
            step=epoch,
            overwrite=True,
        )
        print(f"Saved checkpoint to {ckpt_path}")

        if val_metrics["accuracy"] > best_acc:
            best_acc = val_metrics["accuracy"]
            best_ckpt_path = os.path.join(CKPT_PATH, "best_checkpoint")
            checkpoints.save_checkpoint(
                ckpt_dir=best_ckpt_path,
                target={"opt": {"target": state.params}},
                step=epoch,
                overwrite=True,
            )
            print(f"New best model! PSNR: {best_acc:.2f}")

print("Training completed!")